In [8]:
# Cell 1: Read the bronze_sales Delta table
df = spark.read.format('delta').load('Tables/dbo/bronze_sales')

# Show the first 10 rows
display(df)


StatementMeta(, 6025d354-4796-40e2-8623-0f171e592f45, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65eea993-5ccd-4f2f-91a0-e37f65e2c7c8)

In [9]:
# Cell 2: All information about the feature columns
df.printSchema()

StatementMeta(, 6025d354-4796-40e2-8623-0f171e592f45, 11, Finished, Available, Finished, False)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- Revenue: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Status: string (nullable = true)



In [10]:
# Cell 3:Transformation 1 - Filter rows
# Keep only active orders with positive revenue
df_filtered  = df.filter(
    (df['Status'] == 'Active') & (df['Revenue'] > 0)
)

# Check how many rows remain
print(f'Original row count: {df.count()}')
print(f'Filtered row count: {df_filtered.count()}')

StatementMeta(, 6025d354-4796-40e2-8623-0f171e592f45, 12, Finished, Available, Finished, False)

Original row count: 1525
Filtered row count: 1268


In [11]:
# Cell 4: Transformation 2 - Rename columns to snake_case
df_renamed = df_filtered \
    .withColumnRenamed('OrderID', 'order_id') \
    .withColumnRenamed('OrderDate', 'order_date') \
    .withColumnRenamed('CustomerName', 'customer_name') \
    .withColumnRenamed('Region', 'region') \
    .withColumnRenamed('ProductCategory', 'product_category') \
    .withColumnRenamed('Revenue', 'revenue') \
    .withColumnRenamed('Quantity', 'quantity') \
    .withColumnRenamed('Status', 'status')

display(df_renamed.limit(5))

StatementMeta(, 6025d354-4796-40e2-8623-0f171e592f45, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dbeaac61-124c-4432-9c1a-03cac508f08c)

In [12]:
# Cell 5: Transformation 3 - Add calculated column
from pyspark.sql.functions import col, round

# Apply a fixed INR to USD conversion rate (example: 1 USD = 94 INR)
EXCHANGE_RATE = 94.5

df_transformed = df_renamed.withColumn(
    'revenue_usd',
    round(col('revenue') / EXCHANGE_RATE, 2)
)

display(df_transformed.select('order_id', 'revenue', 'revenue_usd').limit(10))

StatementMeta(, 6025d354-4796-40e2-8623-0f171e592f45, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dfc3dfab-18c3-4962-aec6-409a0b6c7744)

In [13]:
# Cell 6: Change the datatype of order_date, revenue and qunatity
from pyspark.sql.functions import to_date
df_datatypechange = df_transformed \
    .withColumn("order_date", to_date(col("order_date"), "dd-MM-yyyy")) \
    .withColumn("revenue", col("revenue").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int"))

df_datatypechange.printSchema()

StatementMeta(, 6025d354-4796-40e2-8623-0f171e592f45, 15, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- revenue_usd: double (nullable = true)



In [14]:
# Cell 7: Write the transformed Dataframe as silver_sales Delta table
df_datatypechange.write \
    .format('delta') \
    .mode('overwrite') \
    .option("overwriteSchema","true") \
    .saveAsTable('silver_sales')

StatementMeta(, 6025d354-4796-40e2-8623-0f171e592f45, 16, Finished, Available, Finished, False)